xenium hbc

导入相关包

In [1]:
from pathlib import Path
import warnings
import os
import scanpy as sc
import scib
import numpy as np
import pandas as pd
import sys
import scgpt as scg
import matplotlib.pyplot as plt
import anndata

import time
import torch
from memory_profiler import memory_usage
from datetime import datetime
from functools import wraps

plt.style.context('default')
warnings.simplefilter("ignore", ResourceWarning)

model_dir = Path("/home/cavin/jt/benchmark/models/SCGPT")


def measure_resources(cuda_id: int = 0,task_des:str="task",save_dir:str="/home/cavin/jt/benchmark/experiments/results/run_metric/batch_integrate"):
    """
    测量函数运行时的CPU内存、指定GPU的显存消耗(峰值)和运行时间。
    
    参数:
    cuda_id (int): 需要监控的 GPU 编号，例如传入 0 代表监控 cuda:0
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            
            device = None
            if torch.cuda.is_available():
                try:
                    if not torch.cuda.is_initialized():
                        torch.cuda.init()
                    device = torch.device(f"cuda:{cuda_id}")
                    torch.cuda.reset_peak_memory_stats(device)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 初始化监控失败: {e}")

            start_time = time.time()
            
            mem_usage_mb, result = memory_usage(
                (func, args, kwargs), 
                max_usage=True, 
                retval=True
            )
            
            end_time = time.time()
            execution_time = end_time - start_time
            
           
            mem_usage_gb = mem_usage_mb / 1024.0
            
            allocated_gb = 0.0
            cached_gb = 0.0

            
            if device is not None:
                try:
                    allocated_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                    cached_gb = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 显存统计失败: {e}")

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # ===========================
            # 美观的表格式输出
            # ===========================
            report = f"""
============================================================
RESOURCE USAGE REPORT: {func.__name__}
============================================================
Timestamp                   : {timestamp}
Target GPU ID               : {cuda_id}
Execution Time (Minutes)    : {execution_time/60:.4f}
Execution Time (Seconds)    : {execution_time:.2f}
CPU Peak Memory Usage (GB)  : {mem_usage_gb:.4f}
GPU Peak Allocated Mem (GB) : {allocated_gb:.4f}
GPU Peak Cached Mem (GB)    : {cached_gb:.4f}
CUDA Available              : {torch.cuda.is_available()}
============================================================
"""
            print(report)
            save_file_name = "record.csv"
            full_path = os.path.join(save_dir, save_file_name)
            os.makedirs(save_dir, exist_ok=True)
            new_record = {"Timestamp":timestamp,"Task":task_des,
                          "Target GPU ID ":cuda_id,
                          "Execution Time (Minutes)":execution_time / 60,
                          "Execution Time (Seconds)":execution_time,
                          "CPU Peak Memory Usage (GB)":mem_usage_gb,
                          "GPU Peak Allocated Mem (GB)":allocated_gb,
                          "GPU Peak Cached Mem (GB)":cached_gb}
            new_data_df = pd.DataFrame([new_record])
            if not os.path.isfile(full_path):
                # 如果文件不存在：直接正常写入（默认 mode='w'），自带表头
                new_data_df.to_csv(full_path, index=False)
                print(f"文件不存在，已创建新文件并写入表头：{full_path}")
            else:
                # 如果文件已存在：使用追加模式 (mode='a')，并且绝对不写表头 (header=False)
                new_data_df.to_csv(full_path, mode='a', header=False, index=False)
                print(f"文件已存在，数据已成功追加至末尾。")
            return result
        return wrapper
    return decorator

读取数据和数据预处理

In [2]:
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1.h5ad'
adata = sc.read_h5ad(simple_path)
gene_col = "gene_name"
cell_type_key = "cell_type"
celltype_id_labels = adata.obs[cell_type_key].astype("category").cat.codes.values
adata = adata[celltype_id_labels >= 0]
org_adata = adata.copy()
adata

View of AnnData object with n_obs × n_vars = 286532 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [3]:
sc.pp.filter_cells(adata,min_genes=10)
adata.write_h5ad("/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad")
adata

/home/cavin/anaconda3/envs/scgpt/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:176: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["n_genes"] = number


AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [4]:
adata.var_names

Index(['ABCC11', 'ACTA2', 'ACTG2', 'ADAM9', 'ADGRE5', 'ADH1B', 'ADIPOQ',
       'AGR3', 'AHSP', 'AIF1',
       ...
       'TUBB2B', 'TYROBP', 'UCP1', 'USP53', 'VOPP1', 'VWF', 'WARS', 'ZEB1',
       'ZEB2', 'ZNF562'],
      dtype='object', length=313)

In [5]:
adata_idlist = adata.var.ensemble_id.values

In [6]:
adata.var['ensembl_id'] = pd.DataFrame(adata_idlist, index=adata.var.index)
adata.var[gene_col] = adata.var.index
adata.var.index = adata_idlist
adata.var_names.name = None
adata.var_names

Index(['ENSG00000121270', 'ENSG00000107796', 'ENSG00000163017',
       'ENSG00000168615', 'ENSG00000123146', 'ENSG00000196616',
       'ENSG00000181092', 'ENSG00000173467', 'ENSG00000169877',
       'ENSG00000204472',
       ...
       'ENSG00000137285', 'ENSG00000011600', 'ENSG00000109424',
       'ENSG00000145390', 'ENSG00000154978', 'ENSG00000110799',
       'ENSG00000140105', 'ENSG00000148516', 'ENSG00000169554',
       'ENSG00000171466'],
      dtype='object', length=313)

提取嵌入

In [7]:
@measure_resources(task_des="SCGpt run xenium human breast cancer")
def get_emb():
    embed_adata = scg.tasks.embed_data(
        adata,
        model_dir,
        gene_col=gene_col,
        batch_size=16,
    )
    return embed_adata.obsm["X_scGPT"]

In [8]:
emb = get_emb()

scGPT - INFO - match 306/313 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 17612/17612 [01:48<00:00, 162.13it/s]



RESOURCE USAGE REPORT: get_emb
Timestamp                   : 2026-03-04 21:18:04
Target GPU ID               : 0
Execution Time (Minutes)    : 1.8347
Execution Time (Seconds)    : 110.08
CPU Peak Memory Usage (GB)  : 4.2735
GPU Peak Allocated Mem (GB) : 0.2836
GPU Peak Cached Mem (GB)    : 0.3301
CUDA Available              : True

文件不存在，已创建新文件并写入表头：/home/cavin/jt/benchmark/experiments/results/run_metric/batch_integrate/record.csv


/home/cavin/anaconda3/envs/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


保存嵌入

In [9]:
emb.shape

(281780, 512)

In [10]:
emb_df = pd.DataFrame(
    emb, 
    index=adata.obs_names,
    columns=[f"scGPT_dim_{i}" for i in range(emb.shape[1])]
)
save_path = "/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/scgpt_human_breast_cancer_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")

读取嵌入

In [11]:
loaded_emb_df = pd.read_parquet(save_path)
simple_path = "/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad"
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [12]:
adata.obsm["X_scGPT"] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_scGPT'